## **AlphaEarth — ML Training Pipeline**

This notebook trains machine learning models from GEE embedding CSV data, evaluates them, and compares the results.

### **Overall Workflow**:

1. Install dependencies and mount Drive
2. Configure all parameters via `Config`
3. `DataManager`   → Data loading, preprocessing, splitting
4. `ModelRegistry` → Model and GridSearch definitions
5. `Trainer`       → GridSearch, sample-size testing, training, evaluation
6. `Reporter`      → Metrics table, comparison plots, CSV output

In [ ]:
!pip install shap xgboost lightgbm joblib -q

from google.colab import drive
drive.mount('/gdrive')

In [ ]:
import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

warnings.filterwarnings("ignore")

from sklearn.model_selection    import GroupShuffleSplit, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.ensemble           import RandomForestClassifier
from sklearn.svm                import LinearSVC  # SVC
from sklearn.tree               import DecisionTreeClassifier
from sklearn.naive_bayes        import GaussianNB
from sklearn.metrics            import (accuracy_score, f1_score, confusion_matrix,
                                         classification_report, precision_score,
                                         recall_score, cohen_kappa_score,
                                         matthews_corrcoef, roc_auc_score,
                                         balanced_accuracy_score)
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
import shap

import builtins

def print(msg, *args):
    try:
        text = msg % args if args else msg
    except Exception:
        text = str(msg) + " " + " ".join(str(a) for a in args)
    builtins.print(text, flush=True)

### **1. CONFIGURATION**

All paths, parameters, and label settings are centralized in one place. For a different dataset or experiment, simply update this class.

In [ ]:
class Config:
    CSV_PATH       = '/gdrive/MyDrive/alphaearth.csv'
    CLASS_COLUMN   = 'Class'
    POLYGON_COLUMN = 'polygon_id'
    DROP_COLUMNS   = ['CRS']

    CLASS_LABEL_MAP = {
        '10' : 'Hazelnut',
        '20' : 'Forest',
        '30' : 'Permanent Cropland',
        '50' : 'Grassland',
        '60' : 'Sparsely Vegetated',
        '70' : 'Arable Land',
        '80' : 'Urban',
        '90' : 'Road and Rail',
        '100': 'Water Course',
        '110': 'Water Bodies',
        '120': 'Wetland',
    }

    TEST_SIZE  = 0.10
    VAL_SIZE   = 0.20

    GRIDSEARCH_SAMPLES_PER_CLASS = 1000

    TRAIN_SAMPLES_PER_CLASS = None

    CORR_THRESHOLD = None

    SAMPLE_SIZE_TEST_SIZES = [5_000, 15_000, 30_000]

    OUTPUT_DIR   = '/gdrive/MyDrive/AlphaEarth/'
    RANDOM_STATE = 42

    FIGURE_DPI = 300

### **2. DATA MANAGEMENT**

`DataManager` consolidates CSV loading, NaN cleaning, correlation filtering, `LabelEncoder`, `StandardScaler`, and polygon-based train/val/test splitting into a single class. Artifacts (scaler, encoder, column list) are automatically saved to `OUTPUT_DIR`.

In [ ]:
class DataManager:

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load_and_prepare(self):
        print("=" * 60)
        print("Veri yükleniyor: %s", self.cfg.CSV_PATH)

        df = pd.read_csv(self.cfg.CSV_PATH)
        print("Boyut: %d satır × %d kolon", df.shape[0], df.shape[1])
        print("Sınıf dağılımı:\n%s", df[self.cfg.CLASS_COLUMN].value_counts().to_string())

        if self.cfg.POLYGON_COLUMN not in df.columns:
            raise ValueError(
                f"'{self.cfg.POLYGON_COLUMN}' kolonu bulunamadı. "
                "gee_embedding_pipeline.py'nin güncel versiyonunu çalıştır."
            )
        polygon_ids = df[self.cfg.POLYGON_COLUMN].values

        drop = self.cfg.DROP_COLUMNS + [self.cfg.POLYGON_COLUMN]
        df.drop(columns=[c for c in drop if c in df.columns], inplace=True)

        before = len(df)
        mask   = df.notna().all(axis=1)
        df     = df[mask]
        polygon_ids = polygon_ids[mask.values]
        removed = before - len(df)
        if removed:
            print("%d NaN satırı çıkarıldı.", removed)

        feature_cols = [c for c in df.columns if c != self.cfg.CLASS_COLUMN]
        X     = df[feature_cols].copy()
        y_raw = df[self.cfg.CLASS_COLUMN].copy()

        if self.cfg.CLASS_LABEL_MAP:
            y_raw = y_raw.astype(str).map(
                {str(k): v for k, v in self.cfg.CLASS_LABEL_MAP.items()}
            ).fillna(y_raw.astype(str))

        print("Özellik sayısı (ham): %d", X.shape[1])

        if self.cfg.CORR_THRESHOLD is not None:
            X = self._remove_high_corr_features(X, self.cfg.CORR_THRESHOLD)
            print("Özellik sayısı (korelasyon filtresi sonrası): %d", X.shape[1])

        label_encoder = LabelEncoder()
        y             = label_encoder.fit_transform(y_raw)
        class_names   = [str(c) for c in label_encoder.classes_]

        print("Sınıflar: %s", class_names)
        print("Benzersiz poligon sayısı: %d", len(np.unique(polygon_ids)))

        scaler   = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

        os.makedirs(self.cfg.OUTPUT_DIR, exist_ok=True)
        joblib.dump(scaler,          f"{self.cfg.OUTPUT_DIR}scaler.pkl")
        joblib.dump(label_encoder,   f"{self.cfg.OUTPUT_DIR}label_encoder.pkl")
        joblib.dump(list(X.columns), f"{self.cfg.OUTPUT_DIR}feature_columns.pkl")
        print("Artifact'lar kaydedildi → %s", self.cfg.OUTPUT_DIR)

        return X_scaled, y, class_names, label_encoder, scaler, polygon_ids

    def polygon_split(self, X, y, polygon_ids):
        gss_test = GroupShuffleSplit(n_splits=1, test_size=self.cfg.TEST_SIZE,
                                     random_state=self.cfg.RANDOM_STATE)
        trainval_idx, test_idx = next(gss_test.split(X, y, groups=polygon_ids))

        poly_trainval = polygon_ids[trainval_idx]
        gss_val = GroupShuffleSplit(n_splits=1, test_size=self.cfg.VAL_SIZE,
                                    random_state=self.cfg.RANDOM_STATE)
        train_rel, val_rel = next(
            gss_val.split(X.iloc[trainval_idx], y[trainval_idx], groups=poly_trainval)
        )
        train_idx = trainval_idx[train_rel]
        val_idx   = trainval_idx[val_rel]

        n_total = len(y)
        print("Poligon bazlı split özeti:")
        print("  Train : %8d piksel (%%%.0f) | %d poligon",
                 len(train_idx), len(train_idx)/n_total*100,
                 len(np.unique(polygon_ids[train_idx])))
        print("  Val   : %8d piksel (%%%.0f) | %d poligon",
                 len(val_idx), len(val_idx)/n_total*100,
                 len(np.unique(polygon_ids[val_idx])))
        print("  Test  : %8d piksel (%%%.0f) | %d poligon",
                 len(test_idx), len(test_idx)/n_total*100,
                 len(np.unique(polygon_ids[test_idx])))

        train_polys = set(polygon_ids[train_idx])
        val_polys   = set(polygon_ids[val_idx])
        test_polys  = set(polygon_ids[test_idx])
        assert len(train_polys & test_polys) == 0, "Train-Test poligon çakışması!"
        assert len(train_polys & val_polys)  == 0, "Train-Val poligon çakışması!"
        assert len(val_polys   & test_polys) == 0, "Val-Test poligon çakışması!"
        print("Poligon sızıntısı kontrolü: GEÇTI")

        return (X.iloc[train_idx], y[train_idx],
                X.iloc[val_idx],   y[val_idx],
                X.iloc[test_idx],  y[test_idx])

    @staticmethod
    def balanced_sample(X, y, samples_per_class: int, random_state: int):
        rng     = np.random.default_rng(random_state)
        indices = []
        for cls in np.unique(y):
            cls_idx = np.where(y == cls)[0]
            n = min(samples_per_class, len(cls_idx))
            indices.extend(rng.choice(cls_idx, n, replace=False))
        return X.iloc[indices], y[indices]

    @staticmethod
    def _remove_high_corr_features(X: pd.DataFrame, threshold: float) -> pd.DataFrame:
        corr_matrix = X.corr().abs()
        upper       = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
        if to_drop:
            print("%d yüksek korelasyonlu özellik çıkarıldı: %s%s",
                     len(to_drop), to_drop[:5], '...' if len(to_drop) > 5 else '')
        return X.drop(columns=to_drop)

### **3. MODEL REGISTRY**

In [ ]:
class ModelRegistry:

    @staticmethod
    def get_models_and_grids(random_state: int = 42):
        return [
            (
                'RandomForest',
                RandomForestClassifier(
                    class_weight='balanced',
                    random_state=random_state,
                    n_jobs=8
                ),
                {
                    'n_estimators'   : [100, 200, 300],
                    'max_depth'      : [10, 20, None],
                    'min_samples_leaf': [1, 2],
                    'max_features'   : ['sqrt', 'log2'],
                }
            ),
            (
                'DecisionTree',
                DecisionTreeClassifier(
                    class_weight='balanced',
                    random_state=random_state
                ),
                {
                    'max_depth'        : [10, 20, 30],
                    'min_samples_split': [2, 5],
                    'min_samples_leaf' : [1, 2],
                }
            ),
            (
                'XGBoost',
                XGBClassifier(
                    eval_metric='mlogloss',
                    random_state=random_state,
                    n_jobs=8,
                    verbosity=0
                ),
                {
                    'n_estimators'   : [100, 200],
                    'learning_rate'  : [0.05, 0.1],
                    'max_depth'      : [4, 6],
                    'subsample'      : [0.8, 1.0],
                    'colsample_bytree': [0.6, 0.8],
                }
            ),
            (
                'LightGBM',
                LGBMClassifier(
                    class_weight='balanced',
                    random_state=random_state,
                    n_jobs=8,
                    verbosity=-1
                ),
                {
                    'n_estimators' : [100, 200],
                    'learning_rate': [0.05, 0.1],
                    'num_leaves'   : [31, 63],
                    'max_depth'    : [-1, 10],
                }
            ),
            (
                'LinearSVC',
                LinearSVC(
                    class_weight='balanced',
                    random_state=random_state,
                    dual=False,
                    max_iter=5000
                ),
                {
                    'C': [0.1, 1, 10],
                }
            ),
        ]

### **4. TRAINING AND EVALUATION**


`Trainer` manages GridSearch, sample-size testing (for all models), and final training + evaluation steps. Confusion matrices are saved individually as PNG files within this class.

In [ ]:
class Trainer:

    def __init__(self, cfg: Config, data_manager: DataManager):
        self.cfg  = cfg

    def run_gridsearch(self, models, X_gs, y_gs, cv: int = 5):
        print("=" * 60)
        print("GridSearchCV başlatılıyor — %d örnek | %d-fold stratified", len(y_gs), cv)
        print("=" * 60)

        skf         = StratifiedKFold(n_splits=cv, shuffle=True,
                                      random_state=self.cfg.RANDOM_STATE)
        best_models = []

        for name, model, param_grid in models:
            print("GridSearch: %s ...", name)
            t0  = time.time()
            clf = GridSearchCV(
                estimator=model, param_grid=param_grid,
                cv=skf, scoring='f1_weighted',
                n_jobs=8, pre_dispatch='2*n_jobs'
            )
            clf.fit(X_gs, y_gs)
            elapsed = time.time() - t0
            print("  En iyi params : %s", clf.best_params_)
            print("  CV F1 skoru   : %.4f  (%.1fs)", clf.best_score_, elapsed)
            best_models.append((name, clf.best_estimator_, clf.best_params_))

        return best_models

    def run_sample_size_test(self, best_models, X_train, y_train,
                             X_val, y_val, class_names, output_dir):
        print("=" * 60)
        print("Sample-size testi başlatılıyor — boyutlar: %s",
                 self.cfg.SAMPLE_SIZE_TEST_SIZES)
        print("=" * 60)

        records = []

        for name, clf, _ in best_models:
            print("  Model: %s", name)
            for n in self.cfg.SAMPLE_SIZE_TEST_SIZES:
                X_sub, y_sub = DataManager.balanced_sample(
                    X_train, y_train, n, self.cfg.RANDOM_STATE
                )
                actual_n = len(y_sub)

                t0 = time.time()
                clf.fit(X_sub, y_sub)
                elapsed = time.time() - t0

                y_pred = clf.predict(X_val)
                f1     = f1_score(y_val, y_pred, average='weighted', zero_division=0)
                acc    = accuracy_score(y_val, y_pred)
                kappa  = cohen_kappa_score(y_val, y_pred)

                print("    n=%-6d → F1=%.4f | Acc=%.4f | Kappa=%.4f | %.1fs",
                         actual_n, f1, acc, kappa, elapsed)
                records.append({
                    'model'       : name,
                    'target_n'    : n,
                    'actual_n'    : actual_n,
                    'val_f1_w'    : round(f1, 4),
                    'val_accuracy': round(acc, 4),
                    'val_kappa'   : round(kappa, 4),
                    'fit_time_s'  : round(elapsed, 2),
                })

        df_ss = pd.DataFrame(records)
        df_ss.to_csv(f"{output_dir}sample_size_test.csv", index=False)
        print("Sample-size sonuçları kaydedildi → sample_size_test.csv")

        self._plot_sample_size_results(df_ss, output_dir)
        return df_ss

    def _plot_sample_size_results(self, df_ss: pd.DataFrame, output_dir: str):
        """
        Her model için veri boyutu vs. F1 eğrisini çizer.
        Makale formatı: siyah-beyaz dostu renkler, büyük font, temiz arka plan.
        """
        palette = ['#1565C0', '#C62828', '#2E7D32', '#6A1B9A', '#E65100', '#00695C']
        models  = df_ss['model'].unique()

        fig, ax = plt.subplots(figsize=(9, 5.5), dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('#FAFAFA')
        ax.set_facecolor('#FAFAFA')

        for i, model in enumerate(models):
            sub   = df_ss[df_ss['model'] == model].sort_values('actual_n')
            color = palette[i % len(palette)]
            ax.plot(sub['actual_n'], sub['val_f1_w'],
                    marker='o', linewidth=2, markersize=7,
                    color=color, label=model, zorder=3)
            for _, row in sub.iterrows():
                ax.annotate(f"{row['val_f1_w']:.3f}",
                            (row['actual_n'], row['val_f1_w']),
                            textcoords='offset points', xytext=(0, 9),
                            ha='center', fontsize=8, color=color, fontweight='bold')

        ax.set_xlabel('Training Samples per Class', fontsize=12, fontweight='bold', labelpad=8)
        ax.set_ylabel('Weighted F1 Score (Validation)', fontsize=12, fontweight='bold', labelpad=8)
        ax.set_title('AlphaEarth | Effect of Training Data Size on Model Performance',
                     fontsize=13, fontweight='bold', pad=14)
        ax.legend(fontsize=10, framealpha=0.95, edgecolor='#dddddd',
                  loc='lower right', frameon=True)
        ax.set_ylim(0, 1.08)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
        ax.tick_params(axis='both', labelsize=10)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(0.8)
        ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5, zorder=0)

        plt.tight_layout()
        plt.savefig(f"{output_dir}sample_size_comparison.png",
                    bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()

    @staticmethod
    def compute_metrics(y_true, y_pred, y_prob, split_name):
        """
        Split bazlı (val / test) tüm performans metriklerini hesaplar.
        Tüm değerler 4 ondalık basamakla döner (0.xxxx formatı).
        """
        m = {
            f'{split_name}_accuracy'         : round(accuracy_score(y_true, y_pred), 4),
            f'{split_name}_balanced_accuracy': round(balanced_accuracy_score(y_true, y_pred), 4),
            f'{split_name}_precision_w'      : round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4),
            f'{split_name}_recall_w'         : round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4),
            f'{split_name}_f1_weighted'      : round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
            f'{split_name}_f1_macro'         : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
            f'{split_name}_cohen_kappa'      : round(cohen_kappa_score(y_true, y_pred), 4),
            f'{split_name}_mcc'              : round(matthews_corrcoef(y_true, y_pred), 4),
        }
        if y_prob is not None:
            try:
                m[f'{split_name}_roc_auc_ovr'] = round(
                    roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted'), 4
                )
            except Exception:
                m[f'{split_name}_roc_auc_ovr'] = float('nan')
        return m

    def _save_confusion_matrix(self, cm, cm_norm, class_names,
                                model_name, split_name, output_dir, variant):
        """
        Tek bir CM varyantını (normalized veya counts) ayrı PNG olarak kaydeder.
        variant: 'normalized' veya 'counts'
        """
        n    = len(class_names)
        cmap = LinearSegmentedColormap.from_list('cm_bilim', ['#F7FBFF', '#2171B5', '#08306B'])

        cell     = max(0.9, 9.5 / n)
        fs_cell  = max(6, 11 - n // 3)
        fs_label = max(8, 11 - n // 4)

        fig, ax = plt.subplots(figsize=(n * cell, n * cell * 0.88), dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('white')

        im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1, aspect='auto')

        for i in range(n):
            for j in range(n):
                nv = cm_norm[i, j]
                rv = cm[i, j]
                tc = 'white' if nv > 0.50 else '#1a1a1a'
                if variant == 'normalized':
                    ax.text(j, i, f'{nv:.3f}', ha='center', va='center',
                            fontsize=fs_cell, fontweight='bold', color=tc)
                else:
                    ax.text(j, i - 0.13, f'{nv:.3f}', ha='center', va='center',
                            fontsize=fs_cell, fontweight='bold', color=tc)
                    ax.text(j, i + 0.22, f'n={rv:,}', ha='center', va='center',
                            fontsize=max(5, fs_cell - 2), color=tc, alpha=0.80)

        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(class_names, rotation=40, ha='right', fontsize=fs_label)
        ax.set_yticklabels(class_names, fontsize=fs_label)
        ax.set_xlabel('Predicted Class', fontsize=12, fontweight='bold', labelpad=8)
        ax.set_ylabel('True Class',      fontsize=12, fontweight='bold', labelpad=8)

        split_tr    = 'Validation' if split_name == 'val' else 'Test'
        variant_tr  = 'Normalized' if variant == 'normalized' else 'Sample Count'
        ax.set_title(
            f'{variant_tr} Confusion Matrix — {model_name}  ({split_tr} Set)',
            fontsize=12, fontweight='bold', pad=12
        )

        for x in np.arange(-0.5, n, 1):
            ax.axhline(x, color='white', linewidth=0.5)
            ax.axvline(x, color='white', linewidth=0.5)

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=9)
        cbar.set_label('Ratio', fontsize=10)

        plt.tight_layout()
        fname = f"{output_dir}{model_name}_cm_{split_name}_{variant}.png"
        plt.savefig(fname, bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()
        print("CM kaydedildi: %s", fname)

    def _plot_confusion_matrices(self, cm, cm_norm, class_names,
                                  model_name, split_name, output_dir):
        """
        Normalized ve Counts CM'lerini ayrı ayrı kaydeder.
        Ayrıca model klasörüne de kopyalar (tek model çıktı yapısı için).
        """
        self._save_confusion_matrix(
            cm, cm_norm, class_names, model_name, split_name, output_dir, 'normalized'
        )
        self._save_confusion_matrix(
            cm, cm_norm, class_names, model_name, split_name, output_dir, 'counts'
        )

    def evaluate_classifier(self, clf, X_train, y_train,
                             X_val, y_val, X_test, y_test,
                             class_names, output_dir):
        """
        Modeli eğitir; val ve test setlerinde tüm metrikleri hesaplar,
        CM'leri ayrı PNG olarak kaydeder, feature importance grafiği çizer.
        """
        model_name = clf.__class__.__name__
        print("-" * 55)
        print("Eğitiliyor: %s", model_name)

        t0       = time.time()
        clf.fit(X_train, y_train)
        fit_time = time.time() - t0

        results = {'model': model_name, 'fit_time': round(fit_time, 2)}

        for split_name, X_eval, y_eval in [('val', X_val, y_val), ('test', X_test, y_test)]:
            y_pred = clf.predict(X_eval)
            y_prob = clf.predict_proba(X_eval) if hasattr(clf, 'predict_proba') else None
            m      = self.compute_metrics(y_eval, y_pred, y_prob, split_name)
            results.update(m)

            print("[%s] Accuracy=%.4f | BalAcc=%.4f | F1w=%.4f | F1mac=%.4f | "
                     "Kappa=%.4f | MCC=%.4f",
                     split_name.upper(),
                     m[f'{split_name}_accuracy'],
                     m[f'{split_name}_balanced_accuracy'],
                     m[f'{split_name}_f1_weighted'],
                     m[f'{split_name}_f1_macro'],
                     m[f'{split_name}_cohen_kappa'],
                     m[f'{split_name}_mcc'])

            report_dict = classification_report(
                y_eval, y_pred, target_names=class_names,
                digits=4, output_dict=True
            )
            pd.DataFrame(report_dict).transpose().to_csv(
                f"{output_dir}{model_name}_classification_report_{split_name}.csv"
            )

            cm      = confusion_matrix(y_eval, y_pred)
            cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
            self._plot_confusion_matrices(
                cm, cm_norm, class_names, model_name, split_name, output_dir
            )

        print("Eğitim süresi: %.1fs", fit_time)

        if hasattr(clf, 'feature_importances_'):
            fi_df = pd.DataFrame({
                'Feature'   : X_train.columns,
                'Importance': clf.feature_importances_
            }).sort_values('Importance', ascending=False)
            fi_df.to_csv(f"{output_dir}{model_name}_feature_importances.csv", index=False)
            self._plot_feature_importance(fi_df, model_name, output_dir)

        joblib.dump(clf, f"{output_dir}{model_name}_model.pkl")
        return results

    def _plot_feature_importance(self, fi_df, model_name, output_dir):
        """
        Top-20 feature importance yatay bar grafiği.
        Makale formatı: temiz arka plan, büyük etiketler.
        """
        top20 = fi_df.head(20)

        max_val = top20['Importance'].max()
        colors  = [plt.cm.Blues(0.4 + 0.6 * (v / max_val)) for v in top20['Importance'][::-1]]

        fig, ax = plt.subplots(figsize=(9, 6.5), dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('white')
        ax.set_facecolor('#FAFAFA')

        bars = ax.barh(top20['Feature'][::-1], top20['Importance'][::-1],
                       color=colors, edgecolor='white', linewidth=0.6, height=0.65)
        for bar, val in zip(bars, top20['Importance'][::-1]):
            ax.text(bar.get_width() + max_val * 0.01,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=8.5, color='#333333')

        ax.set_xlabel('Importance Score', fontsize=12, fontweight='bold', labelpad=8)
        ax.set_title(f'AlphaEarth | Top 20 Feature Importances — {model_name}',
                     fontsize=13, fontweight='bold', pad=12)
        ax.tick_params(axis='both', labelsize=10)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(0.8)
        ax.grid(axis='x', linestyle='--', linewidth=0.6, alpha=0.5, zorder=0)
        ax.set_xlim(0, max_val * 1.15)
        plt.tight_layout()
        plt.savefig(f"{output_dir}{model_name}_feature_importances.png",
                    bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()

### **5. SHAP ANALYSIS**



`SHAPAnalyzer` performs class-based feature contribution analysis using `TreeExplainer` for tree-based models. The `class_names` list is derived from the CSV — it is not hardcoded.


In [ ]:
class SHAPAnalyzer:

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def run(self, model, X_test, y_test, class_names, output_dir,
            samples_per_class: int = 50):

        model_name = model.__class__.__name__
        shap_dir   = os.path.join(output_dir, 'SHAP') + os.sep
        os.makedirs(shap_dir, exist_ok=True)
        print(f"SHAP Analizi Başlatıldı — Model: {model_name}")

        rng     = np.random.default_rng(42)
        indices = []
        for cls in np.unique(y_test):
            cls_idx = np.where(y_test == cls)[0]
            n = min(samples_per_class, len(cls_idx))
            indices.extend(rng.choice(cls_idx, n, replace=False))

        X_sub = X_test.iloc[indices].reset_index(drop=True)

        try:
            if "LinearSVC" in model_name or "SVC" in model_name:
                explainer = shap.Explainer(model, X_sub)
            else:
                explainer = shap.TreeExplainer(model)

            shap_values = explainer.shap_values(X_sub)
        except Exception as e:
            print(f"Genel Explainer başarısız oldu, KernelExplainer deneniyor (Yavaş olabilir): {e}")
            explainer = shap.KernelExplainer(model.predict, shap.sample(X_sub, 10))
            shap_values = explainer.shap_values(X_sub)

        np.save(f"{shap_dir}{model_name}_shap_values.npy", shap_values)

        n_classes = len(class_names)

        if isinstance(shap_values, list):
            get_sv = lambda i: shap_values[i]
        else:
            get_sv = lambda i: shap_values[:, :, i]

        for i, cls_name in enumerate(class_names):
            sv = get_sv(i)
            plt.figure(figsize=(8, 6), dpi=300)
            shap.summary_plot(sv, features=X_sub,
                              feature_names=X_sub.columns.tolist(),
                              max_display=20, show=False)
            plt.title(f"{cls_name} — {model_name}", fontsize=13, fontweight='bold')
            plt.tight_layout()
            safe_name = cls_name.replace(' ', '_').replace('/', '_')
            plt.savefig(f"{shap_dir}shap_summary_{model_name}_{safe_name}.png",
                        bbox_inches='tight')
            plt.show()
            plt.close()

        ncols = 2
        nrows = (n_classes + 1) // 2

        fig = plt.figure(figsize=(14 * ncols / 2, 5.5 * nrows),
                        dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('white')

        gs = fig.add_gridspec(nrows, ncols, hspace=0.35, wspace=0.25)

        for i, cls_name in enumerate(class_names):
            sv = get_sv(i)

            if i == n_classes - 1 and n_classes % 2 == 1:
                ax = fig.add_subplot(gs[-1, :])
            else:
                r, c = divmod(i, ncols)
                ax = fig.add_subplot(gs[r, c])

            plt.sca(ax)
            shap.summary_plot(
                sv,
                features=X_sub,
                feature_names=X_sub.columns.tolist(),
                max_display=10,
                show=False,
                plot_size=None,
                color_bar=False
            )

            ax.set_title(cls_name, fontsize=13, fontweight='bold', pad=8)
            ax.tick_params(labelsize=10)

        fig.suptitle(
            f'AlphaEarth | SHAP Summary — All Classes (Top 10)\n{model_name}',
            fontsize=18, fontweight='bold', y=0.92
        )

        plt.tight_layout(rect=[0, 0.07, 1, 0.97])

        cbar_ax = fig.add_axes([0.08, 0.062, 0.84, 0.008])

        gradient = np.linspace(0, 1, 512).reshape(1, -1)
        cmap = mpl.colors.LinearSegmentedColormap.from_list(
            "custom_shap",
            ["#008afb", "#ff0051"]
        )
        cbar_ax.imshow(gradient, aspect='auto', cmap=cmap)
        cbar_ax.set_axis_off()

        fig.text(0.08, 0.045, "Low", ha='left', va='top', fontsize=12)
        fig.text(0.92, 0.045, "High", ha='right', va='top', fontsize=12)

        fig.text(0.5, 0.035, "Feature Value", ha='center', va='top', fontsize=16)

        plt.savefig(
            f"{shap_dir}shap_summary_{model_name}_all_classes.png",
            bbox_inches='tight',
            dpi=self.cfg.FIGURE_DPI
        )
        plt.show()
        plt.close()

        if isinstance(shap_values, list):
            sv_mean = np.mean([np.abs(get_sv(i)) for i in range(n_classes)], axis=0)
        else:
            sv_mean = np.mean(np.abs(shap_values), axis=2)

        mean_abs = sv_mean.mean(axis=0)
        top_n    = min(20, len(mean_abs))
        fi_shap  = pd.Series(mean_abs, index=X_sub.columns).sort_values(ascending=True).tail(top_n)

        fig, ax = plt.subplots(figsize=(10, 7), dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('white')
        ax.set_facecolor('#FAFAFA')

        max_val = fi_shap.max()
        colors  = [plt.cm.RdYlBu_r(0.2 + 0.75 * (v / max_val)) for v in fi_shap.values]

        bars = ax.barh(fi_shap.index, fi_shap.values,
                       color=colors, edgecolor='white', linewidth=0.6,
                       height=0.65, zorder=3)

        for bar, val in zip(bars, fi_shap.values):
            ax.text(bar.get_width() + max_val * 0.015,
                    bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', ha='left',
                    fontsize=8.5, color='#333333', fontweight='bold')

        sm = plt.cm.ScalarMappable(cmap='RdYlBu_r',
                                    norm=plt.Normalize(vmin=fi_shap.min(),
                                                       vmax=fi_shap.max()))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
        cbar.set_label('Mean |SHAP Value|', fontsize=9, labelpad=8)
        cbar.ax.tick_params(labelsize=8)

        ax.axvline(fi_shap.mean(), color='#E65100', linewidth=1.2,
                   linestyle='--', alpha=0.7, zorder=2,
                   label=f'Mean = {fi_shap.mean():.4f}')

        ax.set_title(f'AlphaEarth | Mean |SHAP| — Top {top_n} Features\n{model_name}',
                     fontsize=13, fontweight='bold', pad=14)
        ax.set_xlabel('Mean |SHAP Value|', fontsize=11, fontweight='bold', labelpad=8)
        ax.set_xlim(0, max_val * 1.22)
        ax.tick_params(axis='both', labelsize=9.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_linewidth(0.8)
        ax.grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.45, zorder=0)
        ax.legend(fontsize=9, framealpha=0.9, edgecolor='#dddddd', loc='lower right')

        plt.tight_layout()
        plt.savefig(f"{shap_dir}shap_mean_abs_{model_name}.png",
                    bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()

### **6. REPORT**

Reporter saves the metrics of all models as both individual and combined CSV files, and produces article-quality comparison charts. The charts feature font sizes, grids, and color palettes that conform to article standards.

In [ ]:
class Reporter:

    METRIC_LABELS = {
        'test_f1_weighted'      : 'F1 Weighted',
        'test_f1_macro'         : 'F1 Macro',
        'test_balanced_accuracy': 'Balanced Accuracy',
        'test_cohen_kappa'      : 'Cohen Kappa',
        'test_mcc'              : 'MCC',
        'test_accuracy'         : 'Overall Accuracy',
    }

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def save_individual_results(self, results: list, output_dir: str):
        for res in results:
            model_name = res['model']
            model_dir  = os.path.join(output_dir, model_name)
            os.makedirs(model_dir, exist_ok=True)
            pd.DataFrame([res]).to_csv(
                f"{model_dir}/{model_name}_metrics.csv", index=False
            )
            print("Bireysel metrikler kaydedildi: %s", model_dir)

    def save_combined_results(self, results: list, output_dir: str):
        df = pd.DataFrame(results)
        numeric_cols = df.select_dtypes(include='number').columns
        df[numeric_cols] = df[numeric_cols].round(4)
        df.sort_values('test_f1_weighted', ascending=False, inplace=True)
        path = f"{output_dir}all_models_combined_results.csv"
        df.to_csv(path, index=False)
        print("Birleşik sonuçlar kaydedildi: %s", path)
        return df

    def plot_model_comparison(self, results: list, output_dir: str):
        df_res = pd.DataFrame(results)

        print("=" * 60)
        print("MODEL KARŞILAŞTIRMASI (Test Seti)")
        print("=" * 60)
        display_cols = ['model', 'test_f1_weighted', 'test_balanced_accuracy',
                        'test_cohen_kappa', 'test_mcc', 'fit_time']
        display_cols = [c for c in display_cols if c in df_res.columns]
        print("\n%s", df_res[display_cols].sort_values(
            'test_f1_weighted', ascending=False).to_string(index=False))

        metrics = [(k, v) for k, v in self.METRIC_LABELS.items() if k in df_res.columns]
        n_m     = len(metrics)
        ncols   = min(3, n_m)
        nrows   = (n_m + ncols - 1) // ncols

        palette = ['#1565C0', '#C62828', '#2E7D32', '#6A1B9A', '#E65100', '#00695C']

        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(5.8 * ncols, 4.8 * nrows),
                                  dpi=self.cfg.FIGURE_DPI)
        fig.patch.set_facecolor('white')
        axes = np.array(axes).flatten()

        for idx, (metric, label) in enumerate(metrics):
            ax   = axes[idx]
            df_s = df_res.sort_values(metric, ascending=False)
            ax.set_facecolor('#FAFAFA')

            bars = ax.bar(df_s['model'], df_s[metric],
                          color=[palette[i % len(palette)] for i in range(len(df_s))],
                          edgecolor='white', linewidth=0.8, width=0.52, zorder=3)

            for bar in bars:
                h = bar.get_height()
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.007,
                        f'{h:.3f}', ha='center', va='bottom',
                        fontsize=9, fontweight='bold', color='#222222')

            ax.set_title(label, fontsize=12, fontweight='bold', pad=8)
            ax.set_ylabel(label, fontsize=10)
            ax.set_ylim(0, 1.13)
            ax.set_xticklabels(df_s['model'], rotation=32, ha='right', fontsize=9.5)
            ax.tick_params(axis='y', labelsize=9.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.spines[['left', 'bottom']].set_linewidth(0.7)
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
            ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)

        for idx in range(len(metrics), len(axes)):
            axes[idx].set_visible(False)

        fig.suptitle('AlphaEarth | Model Performance Comparison — Test Set',
                     fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        plt.savefig(f"{output_dir}model_comparison.png",
                    bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()
        print("Karşılaştırma grafiği kaydedildi: model_comparison.png")

    def plot_radar_chart(self, results: list, output_dir: str):

        df_res  = pd.DataFrame(results)
        metrics = [k for k in self.METRIC_LABELS if k in df_res.columns]
        labels  = [self.METRIC_LABELS[m] for m in metrics]
        n       = len(metrics)
        if n < 3:
            return

        angles  = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
        angles += angles[:1]

        palette = ['#1565C0', '#C62828', '#2E7D32', '#6A1B9A', '#E65100', '#00695C']

        fig, ax = plt.subplots(figsize=(8, 8), dpi=self.cfg.FIGURE_DPI,
                               subplot_kw=dict(polar=True))
        fig.patch.set_facecolor('white')

        for i, row in df_res.iterrows():
            values  = [row[m] for m in metrics]
            values += values[:1]
            color   = palette[i % len(palette)]
            ax.plot(angles, values, linewidth=2.2, color=color,
                    label=row['model'], zorder=3)
            ax.fill(angles, values, alpha=0.07, color=color)
            for ang, val in zip(angles[:-1], values[:-1]):
                ax.text(ang, val + 0.04, f'{val:.2f}', ha='center', va='center',
                        fontsize=7.5, color=color, fontweight='bold')

        ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10.5)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.20', '0.40', '0.60', '0.80', '1.00'], fontsize=8.5,
                           color='#666666')
        ax.set_title('AlphaEarth | Model Performance Profile — Test Set',
                     fontsize=13, fontweight='bold', pad=22)
        ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.18), fontsize=10,
                  framealpha=0.95, edgecolor='#dddddd')
        ax.grid(linestyle='--', linewidth=0.6, alpha=0.6)
        ax.spines['polar'].set_linewidth(0.8)

        plt.tight_layout()
        plt.savefig(f"{output_dir}model_comparison_radar.png",
                    bbox_inches='tight', dpi=self.cfg.FIGURE_DPI)
        plt.show()
        plt.close()
        print("Radar grafiği kaydedildi: model_comparison_radar.png")

    def print_summary_table(self, df_combined: pd.DataFrame):
        cols = ['model'] + [k for k in self.METRIC_LABELS if k in df_combined.columns] + ['fit_time']
        cols = [c for c in cols if c in df_combined.columns]
        print("Özet Tablo:\n%s", df_combined[cols].to_string(index=False))

### **7. RUN PIPELINE**

After editing the configuration, run this cell. The pipeline will run in the following order: data preparation → split → GridSearch → sample-size test → final training → reporting.

In [ ]:
cfg          = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

data_manager      = DataManager(cfg)
trainer           = Trainer(cfg, data_manager)
reporter          = Reporter(cfg)
shap_analyzer     = SHAPAnalyzer(cfg)

In [ ]:
# ── Step 1: Data Preparation ─────────────────────────────────────────────────
X, y, class_names, label_encoder, scaler, polygon_ids = data_manager.load_and_prepare()

In [ ]:
# ── Step 2: Polygon-Based Split ────────────────────────────────────────────
X_train, y_train, X_val, y_val, X_test, y_test = data_manager.polygon_split(
    X, y, polygon_ids
)

In [ ]:
# ── Step 3: Balanced Sampling for GridSearch ──────────────────────────────
X_gs, y_gs = DataManager.balanced_sample(
    X_train, y_train,
    cfg.GRIDSEARCH_SAMPLES_PER_CLASS,
    cfg.RANDOM_STATE
)

In [ ]:
# ── Adım 4: GridSearch ─────────────────────────────────────────────────────
models      = ModelRegistry.get_models_and_grids(cfg.RANDOM_STATE)
best_models = trainer.run_gridsearch(models, X_gs, y_gs)

In [ ]:
# ── Step 5: Sample-Size Test (all models) ───────────────────────────────
# Using the best parameters from GridSearch, it measures the effect of data volume (SAMPLE_SIZE_TEST_SIZES).
trainer.run_sample_size_test(
    best_models, X_train, y_train, X_val, y_val,
    class_names, cfg.OUTPUT_DIR
)

In [ ]:
# ── Adım 6: Final Training ───────────────────────────────────────────────────
if cfg.TRAIN_SAMPLES_PER_CLASS:
    X_tr, y_tr = DataManager.balanced_sample(
        X_train, y_train,
        cfg.TRAIN_SAMPLES_PER_CLASS,
        cfg.RANDOM_STATE
    )
else:
    X_tr, y_tr = X_train, y_train

results = []
for name, best_clf, _ in best_models:
    model_dir = os.path.join(cfg.OUTPUT_DIR, name) + os.sep
    os.makedirs(model_dir, exist_ok=True)

    res = trainer.evaluate_classifier(
        best_clf, X_tr, y_tr,
        X_val, y_val, X_test, y_test,
        class_names, model_dir
    )
    results.append(res)
    reporter.save_individual_results([res], model_dir)

In [ ]:
# ── Adım 7: Report ──────────────────────────────────────────────────────
# Merged CSV → for comparison
df_combined = reporter.save_combined_results(results, cfg.OUTPUT_DIR)
reporter.print_summary_table(df_combined)

In [ ]:
reporter.plot_model_comparison(results, cfg.OUTPUT_DIR)
reporter.plot_radar_chart(results, cfg.OUTPUT_DIR)

In [ ]:
# ── Step 8: SHAP (tree-based best model) ───────────────────────────────
_shap_done = False
for _name, _clf, _ in sorted(
        best_models,
        key=lambda t: f1_score(y_test, t[1].predict(X_test), average='weighted'),
        reverse=True):
    try:
        shap_analyzer.run(_clf, X_test, y_test, class_names, cfg.OUTPUT_DIR)
        print("SHAP tamamlandı — model: %s", _name)
        _shap_done = True
        break
    except Exception as e:
        print("SHAP atlandı (%s): %s", _name, e)

if not _shap_done:
    print("Hiçbir model SHAP ile uyumlu değil, adım atlandı.")

print("Pipeline tamamlandı. Çıktılar: %s", cfg.OUTPUT_DIR)